**학습 파이프라인 개요**  

	1.라벨 TSV를 읽어서 이미지 파일의 전체 경로(path)를 만들고  

	2.HuggingFace Dataset으로 감싸서  
	
	3.DataLoader가 배치 만들 때(collate_fn) 그때그때 이미지 열고 + processor로 전처리 + text를 토큰화해서  

	4.VisionEncoderDecoderModel(TrOCR)에 넣어 loss를 계산하고  

	5.gradient accumulation으로 큰 배치 효과를 내면서  

	6.epoch마다 validation loss를 계산해, 가장 성능이 좋은 시점의 체크포인트를 저장하는 코드

**데이터셋 경로 및 최종모델 저장 경로 수정 필요**

In [ ]:
import os, math, gc, torch
import pandas as pd
from PIL import Image

# Hugging Face datasets: pandas dataframe -> Dataset 형태로 변환할 때 사용
from datasets import Dataset

# PyTorch DataLoader: 배치 단위로 데이터를 꺼내고, 멀티프로세싱으로 로딩 속도 향상
from torch.utils.data import DataLoader

# Optimizer / Gradient 관련
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_

# Hugging Face Transformers: TrOCR 구성요소
# - TrOCRProcessor: 이미지 전처리 + tokenizer(문자 라벨 토큰화) 묶음
# - VisionEncoderDecoderModel: (비전 인코더 + 텍스트 디코더)로 OCR 수행
# - get_scheduler: 학습률 스케줄러 편하게 생성
from transformers import (
    TrOCRProcessor, VisionEncoderDecoderModel, get_scheduler
)

# 진행률 표시
from tqdm.auto import tqdm


# =========================================================
# 1️⃣ 데이터 준비: "이미지 경로(path) + 정답 텍스트(text)"만 메모리에 들고오기
# =========================================================
# - train_labels.tsv, val_labels.tsv에는 보통 (filename \t text) 형태의 라벨이 있음
# - 실제 이미지는 train_dir/val_dir에 존재한다고 가정
train_tsv = "/home/work/Seminar_Folder/A-team/trocr_project/data/train_labels.tsv"
val_tsv   = "/home/work/Seminar_Folder/A-team/trocr_project/data/val_labels.tsv"
train_dir = "/home/work/Seminar_Folder/A-team/trocr_project/data/train"
val_dir   = "/home/work/Seminar_Folder/A-team/trocr_project/data/val"

# TSV 로딩: sep="\t" 필수 (탭 구분)
train_df = pd.read_csv(train_tsv, sep="\t")
val_df   = pd.read_csv(val_tsv, sep="\t")

# filename 컬럼을 이용해 "실제 이미지 파일 경로(path)"를 생성
# 예: train_dir/img_0001.jpg 같은 형태로 만듦
train_df["path"] = train_df["filename"].apply(lambda x: f"{train_dir}/{x}")
val_df["path"]   = val_df["filename"].apply(lambda x: f"{val_dir}/{x}")

# Hugging Face Dataset으로 변환
# 여기서는 "이미지 자체를 메모리에 올리지 않고", 경로+라벨만 저장해 둠
# 실제 이미지는 collate_fn에서 배치 단위로 열게 됨 (메모리 절약)
train_ds = Dataset.from_pandas(train_df[["path", "text"]])
val_ds   = Dataset.from_pandas(val_df[["path", "text"]])

print(f"✅ Dataset ready | Train: {len(train_ds)}, Val: {len(val_ds)}")


# =========================================================
# 2️⃣ TrOCR 모델 및 Processor 설정
# =========================================================
# TrOCR는 "이미지 -> 텍스트" 변환을 위한 Encoder-Decoder 구조임
# - Encoder: 이미지 특징을 추출 (ViT 등 비전 모델)
# - Decoder: 텍스트를 한 글자/토큰씩 생성 (Transformer decoder)
model_name = "microsoft/trocr-small-stage1"

# processor: 이미지 전처리 + 텍스트 토크나이저 포함
processor = TrOCRProcessor.from_pretrained(model_name)

# model: VisionEncoderDecoderModel 형태로 로드
model = VisionEncoderDecoderModel.from_pretrained(model_name)

# forced_bos/eos 토큰 강제 설정을 해제 (특정 케이스에서 생성/학습 꼬임 방지용)
model.config.forced_bos_token_id = None
model.config.forced_eos_token_id = None

# 디코더 시작 토큰(bos), 패딩 토큰(pad), 종료 토큰(eos) 설정
# - decoder_start_token_id: 디코더가 첫 토큰으로 무엇을 넣고 시작할지
# - pad_token_id: padding용 토큰 id
# - eos_token_id: 문장 종료 토큰 id
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.eos_token_id           = processor.tokenizer.eos_token_id

# vocab_size를 디코더 vocab_size와 일치시킴 (일부 구성에서 필요)
model.config.vocab_size             = model.config.decoder.vocab_size

# 생성 최대 길이 (추론 시에도 영향을 줌)
model.config.max_length             = 32

# 학습 중 use_cache=False: 디코더 캐시를 끄면 메모리가 더 안정적 (대신 약간 느려질 수 있음)
model.config.use_cache              = False

# 학습 디바이스 설정: 가능하면 GPU(cuda) 사용
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


# =========================================================
# 3️⃣ AMP(자동 mixed precision) 설정
# =========================================================
# bfloat16 지원 GPU면 bf16이 안정적인 경우가 많음 (특히 A100/H100 등)
# bf16 지원이 없으면 fp16 + GradScaler 사용
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

# bf16이면 보통 GradScaler 필요 없음, fp16이면 overflow 방지를 위해 필요
scaler = None if use_bf16 else torch.amp.GradScaler(device="cuda")

# TF32는 (Ampere 이상) matmul 성능 향상. 약간의 정밀도 손해를 감수하고 속도 향상
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True


# =========================================================
# 4️⃣ collate_fn: DataLoader가 배치를 만들 때 호출되는 함수
# =========================================================
# 기본 DataLoader는 tensor stacking만 하는데,
# 여기서는 "이미지 파일 읽기 -> processor로 전처리 -> tokenizer로 라벨 토큰화"를 해야 함.
def collate_fn(examples):
    imgs, txts = [], []

    # 예외 처리: 이미지가 깨져있거나 path가 잘못된 샘플이 섞여 있을 수 있으므로 try/except
    for ex in examples:
        try:
            # 이미지 로드 후 RGB로 통일
            imgs.append(Image.open(ex["path"]).convert("RGB"))
            # 라벨 텍스트는 문자열로 변환해서 저장
            txts.append(str(ex["text"]))
        except Exception as e:
            # 문제가 있으면 해당 샘플은 배치에서 제외
            continue

    # 만약 배치 내 모든 샘플이 실패하면 "빈 배치"를 반환
    # (학습 루프에서 numel()==0 체크 후 skip)
    if not imgs:
        return {"pixel_values": torch.empty(0), "labels": torch.empty(0, dtype=torch.long)}

    # 이미지 전처리 -> pixel_values 생성 (shape: [B, C, H, W])
    pixel_values = processor(images=imgs, return_tensors="pt").pixel_values

    # 텍스트 라벨 토큰화
    # - padding="max_length": 고정 길이로 패딩
    # - max_length=24: 라벨 최대 길이 제한 (너무 길면 truncation)
    labels = processor.tokenizer(
        txts, padding="max_length", max_length=24, truncation=True, return_tensors="pt"
    ).input_ids

    # CrossEntropyLoss에서 "무시할 토큰"은 -100으로 표시하는 것이 관례
    # padding 토큰을 -100으로 바꿔서 loss 계산에서 제외되도록 함
    labels[labels == processor.tokenizer.pad_token_id] = -100

    return {"pixel_values": pixel_values, "labels": labels}


# =========================================================
# 5️⃣ DataLoader 설정
# =========================================================
# per_device_batch: GPU 1장 기준 배치 사이즈
# grad_accum_steps: gradient accumulation 스텝 수
#   - 예: batch=64, accum=4면 "실제 효과 배치 = 64*4 = 256" 비슷한 효과
per_device_batch  = 64
grad_accum_steps  = 4

# num_workers: 데이터 로딩 멀티프로세싱 개수 (환경에 따라 조절)
num_workers_train = 8
num_workers_val   = 4

# pin_memory: GPU로 데이터를 넘길 때 속도 향상 가능
# persistent_workers: epoch 넘어도 worker 유지 (속도 향상)
# prefetch_factor: worker가 미리 가져올 배치 수 (메모리/성능 트레이드오프)
train_loader = DataLoader(
    train_ds,
    batch_size=per_device_batch,
    shuffle=True,
    num_workers=num_workers_train,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds,
    batch_size=per_device_batch,
    shuffle=False,
    num_workers=num_workers_val,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    collate_fn=collate_fn
)


# =========================================================
# 6️⃣ Optimizer & Scheduler 설정
# =========================================================
num_epochs = 10

# 누적 학습 스텝 수 계산
# - train_loader의 batch step을 grad_accum_steps로 나눈 것이 optimizer.step()이 호출되는 횟수
steps_per_epoch    = math.ceil(len(train_loader) / grad_accum_steps)
num_training_steps = steps_per_epoch * num_epochs

# AdamW: 트랜스포머 계열에서 표준적으로 많이 사용
optimizer = AdamW(model.parameters(), lr=2e-5)

# learning rate scheduler: linear warmup + linear decay
# warmup: 초반 학습 안정화를 위해 조금씩 lr을 올리는 구간
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=int(0.03 * num_training_steps),
    num_training_steps=num_training_steps
)

# validation loss가 가장 낮을 때의 모델을 저장하기 위해 best_val_loss 관리
best_val_loss = float("inf")

# 저장 경로
save_dir = "./models/finetuned_trocr_small_epoch10/best_checkpoint"
os.makedirs(save_dir, exist_ok=True)


# =========================================================
# 7️⃣ 학습 루프 (Train -> Validation -> Best checkpoint 저장)
# =========================================================
for epoch in range(num_epochs):
    # -------------------------
    # (A) Train
    # -------------------------
    model.train()
    running = 0.0  # 누적 train loss 합
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for step, batch in enumerate(pbar):
        # collate_fn에서 빈 배치가 올 수 있으니 방어
        if batch["pixel_values"].numel() == 0:
            continue

        # CPU tensor -> GPU tensor 이동 (non_blocking은 pin_memory=True일 때 도움)
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        # ----- AMP(혼합정밀) 구간 -----
        # 현재 코드에서는 enabled=False라서 AMP를 사용하지 않는 상태임.
        # 원래 의도는 enabled=True로 두고 dtype=amp_dtype로 학습 메모리를 줄이는 것.
        #
        # 안전한 동작을 원하면: enabled=torch.cuda.is_available() 로 켜고,
        # dtype=amp_dtype 로 넣는 형태가 일반적.
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=False):
            outputs = model(**batch)
            loss = outputs.loss

        # ----- 유효 라벨 체크 -----
        # labels == -100 은 loss에서 무시되므로, 전부 -100이면 학습 의미가 없음
        valid_labels = (batch["labels"] != -100).sum().item()
        if valid_labels == 0 or not torch.isfinite(loss):
            # NaN/Inf 또는 라벨이 전부 padding인 경우 skip
            print(f"⚠️ NaN/Inf loss or empty label batch — skip | valid_labels={valid_labels}")
            optimizer.zero_grad(set_to_none=True)
            continue

        # ----- Gradient Accumulation -----
        # 누적을 위해 loss를 grad_accum_steps로 나눠줌 (스케일 유지)
        loss = loss / grad_accum_steps

        # bf16이면 scaler 없이 backward, fp16이면 scaler로 scale 후 backward
        if scaler is None:
            loss.backward()
        else:
            scaler.scale(loss).backward()

        running += loss.item() * grad_accum_steps  # 원래 스케일로 누적 표시

        # ----- Optimizer step -----
        # grad_accum_steps마다 한 번 optimizer.step() 수행
        if (step + 1) % grad_accum_steps == 0:
            if scaler is None:
                # gradient exploding 방지용 clipping
                clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            else:
                # fp16인 경우 scaler 사용: unscale -> clip -> step -> update
                scaler.unscale_(optimizer)
                clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        # 진행바에 평균 loss 표시
        pbar.set_postfix(train_loss=f"{running / max(1, (step+1)):.4f}")

    # epoch 평균 train loss
    avg_train = running / len(train_loader)

    # -------------------------
    # (B) Validation
    # -------------------------
    model.eval()
    val_loss = 0.0
    count = 0  # 유효 배치 개수

    with torch.no_grad():
        for batch in val_loader:
            if batch["pixel_values"].numel() == 0:
                continue

            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            valid_labels = (batch["labels"] != -100).sum().item()
            if valid_labels == 0:
                continue

            # 검증 시에도 autocast를 쓸 수 있지만, 지금은 enabled=False라 실제로는 FP32로 동작
            with torch.autocast("cuda", dtype=torch.float32, enabled=False):
                outputs = model(**batch)
                loss_val = outputs.loss

            if not torch.isfinite(loss_val):
                print(f"⚠️ NaN/Inf val loss — skip | valid_labels={valid_labels}")
                continue

            val_loss += loss_val.item()
            count += 1

    # 원래 의도는 "유효 배치(count)"로 나누는 것이 더 정확함.
    # 하지만 현재 코드는 len(val_loader)로 나누고 있음 (빈 배치를 skip해도 분모는 고정)
    # -> 평가값이 약간 왜곡될 수 있음.
    avg_val = val_loss / len(val_loader)

    print(f"\n📘 Epoch {epoch+1}/{num_epochs} | Train: {avg_train:.4f} | Val: {avg_val:.4f}")

    # GPU 캐시/파이썬 가비지 컬렉션으로 메모리 정리
    torch.cuda.empty_cache()
    gc.collect()

    # -------------------------
    # (C) Best checkpoint 저장
    # -------------------------
    # val_loss가 개선되면 모델 가중치 + processor(토크나이저/전처리 설정) 저장
    if avg_val < best_val_loss and avg_val > 0 and math.isfinite(avg_val):
        best_val_loss = avg_val
        model.save_pretrained(save_dir)
        processor.save_pretrained(save_dir)
        print(f"✅ Best model updated! val_loss={avg_val:.4f}")